<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/867_PCFDv2_Discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Discovery Engine

### Product–Customer Fit Discovery Orchestrator v2 (PCFDv2)

This module is the **core intelligence engine** of the orchestrator.

Once the data has been loaded, the discovery layer analyzes the datasets and surfaces **strategic signals about customers, products, features, and market trends**.

Unlike many AI-driven analytics systems, this discovery engine is intentionally **rules-based**.

All thresholds and decision criteria come from configuration rather than being embedded in opaque algorithms.

This design ensures the system remains:

* explainable
* reproducible
* controllable by business leaders

---

# 🧠 What the Discovery Engine Actually Does

At a high level, this module transforms raw operational data into **structured business intelligence**.

It identifies signals such as:

• high-value customer segments
• high-margin products
• emerging product adoption
• declining product lines
• accelerating feature demand
• expanding market segments
• cross-sell bundle opportunities

These discoveries are organized into **five intelligence layers**.

```text
Customer Intelligence
Product Intelligence
Feature Intelligence
Trend Intelligence
Strategic Signals
```

Separating insights this way keeps the orchestrator **easy to interpret and extend**.

---

# 🎯 Design Philosophy — Rules First

A key design principle of this module is:

> **Discovery logic should be transparent and configurable.**

Instead of embedding decisions in machine learning models, the system evaluates signals using explicit thresholds defined in configuration.

Examples include:

```python
adoption_growth_min_pct
high_margin_min_pct
segment_growth_min_pct
feature_demand_min_requests
opportunity_score_min
```

This ensures discoveries are driven by **business rules rather than unpredictable model behavior**.

For executives evaluating AI systems, this design provides something extremely valuable:

**control over how insights are generated.**

---

# 🔎 Customer Intelligence

The `discover_customer_intel()` function identifies **high-value customer segments**.

It combines:

• customer profiles
• revenue metrics
• lifetime value
• retention risk

Customers are grouped into segments such as:

```
Account_Type
Age_Group
Location_Tier
```

The function calculates metrics such as:

* segment revenue contribution
* average customer lifetime value
* average retention risk

Segments are then ranked by **total revenue contribution**.

This allows the orchestrator to highlight questions such as:

> Which customer segments generate the majority of our revenue?

For product and growth teams, this information helps focus resources on the **most valuable parts of the customer base**.

---

# 📦 Product Intelligence

The `discover_product_intel()` function analyzes product performance.

It identifies products that combine two valuable characteristics:

```
High Profit Margin
+
Strong Usage
```

The system:

1. reads profit margins from product metrics
2. counts product usage from transactions
3. filters products above the configured margin threshold

The output highlights products that represent **profitable adoption opportunities**.

For leadership teams, this surfaces questions like:

> Which products are both popular and highly profitable?

These are often the **best candidates for expansion or bundling**.

---

# ⚙️ Feature Intelligence

The `discover_feature_intel()` function evaluates **product capabilities rather than products themselves**.

It analyzes two signals:

```
Feature Usage
+
Market Feature Requests
```

By combining internal usage data with external market signals, the orchestrator can detect:

• features customers heavily rely on
• features competitors offer
• capabilities the product portfolio may be missing

This helps product leaders answer questions such as:

> Which capabilities are becoming essential for customers?

Feature intelligence is often one of the **most valuable inputs for product roadmap planning**.

---

# 📈 Trend Intelligence

The `discover_trend_intel()` function performs **time-based analysis**.

Instead of evaluating static snapshots, it analyzes how signals change across time.

Examples include:

```
Product adoption growth
Segment expansion
Feature demand acceleration
```

The system computes growth percentages using historical snapshots.

Products can be classified as:

```
Emerging Products
Declining Products
```

Segments can be classified as:

```
Fast-Expanding Segments
```

This transforms the orchestrator from a **reporting system** into a **trend detection engine**.

For strategy teams, this answers critical questions like:

> What is gaining momentum in our market right now?

---

# 🚀 Strategic Signals

The final discovery layer processes **precomputed opportunity signals**.

These signals represent patterns detected across the product ecosystem, such as:

```
Cross-sell opportunities
Product bundles
Expansion paths
```

Each opportunity is evaluated using a configurable score threshold.

Example criteria:

```
Opportunity_Score ≥ configured threshold
```

Bundles are then ranked by:

```
Opportunity Score
+
Observed Customer Adoption
```

This produces a prioritized list of **revenue expansion opportunities**.

For sales and growth teams, these signals highlight where **additional revenue may already exist inside the customer base**.

---

# 🔄 Coordinating the Discovery Pipeline

The function `run_all_discovery()` orchestrates all discovery layers.

It runs each intelligence module sequentially:

```
Customer Intelligence
Product Intelligence
Feature Intelligence
Trend Intelligence
Strategic Signals
```

Each layer returns structured insights, which are combined into a single discovery output.

This modular structure makes the system easy to:

* extend with additional discovery modules
* debug individual analysis layers
* adjust thresholds without rewriting logic

---

# 🏢 Why Executives Would Value This Architecture

Many AI systems attempt to generate insights using complex models that are difficult to interpret.

This discovery engine takes a different approach.

It produces insights using **transparent analytical logic**.

Executives can immediately understand why a signal was generated.

For example:

```
Emerging Product Detected

Product: P14
Adoption Growth: 32%
Threshold: 10%
```

The reasoning is visible.

This transparency builds confidence that discoveries are grounded in **measurable business signals rather than opaque algorithms**.

---

# ⚠ Potential Enhancements

The discovery engine is already well structured, but several additions could make it even stronger.

### Opportunity Scoring Across Layers

Currently each discovery type is independent.

Introducing a unified **opportunity score** could allow the orchestrator to rank insights across all intelligence layers.

---

### Revenue Impact Estimation

Discoveries could include estimated financial impact such as:

```
estimated_bundle_revenue
segment_expansion_value
```

This would help leadership prioritize opportunities.

---

### Signal Confidence Metrics

Confidence scores could be calculated using signal density, adoption velocity, or cross-dataset agreement.

---

# ⭐ Final Assessment

This discovery module represents the **analytical heart of the orchestrator**.

Its strengths include:

• modular intelligence layers
• configurable discovery thresholds
• structured outputs
• deterministic logic
• explainable business signals

Rather than relying on opaque AI models, the system uses **clear analytical reasoning to surface strategic opportunities**.

This approach produces a discovery engine that is not only powerful but also **trustworthy and controllable by the business**.



In [ ]:
"""
Rules-based discovery: customer, product, feature, trend intelligence and strategic signals.
All thresholds come from config; no hardcoding.
"""

from typing import Any, Dict, List, Optional


def _float(x: Any, default: float = 0.0) -> float:
    try:
        return float(x) if x is not None and x != "" else default
    except (TypeError, ValueError):
        return default


def _int(x: Any, default: int = 0) -> int:
    try:
        return int(float(x)) if x is not None and x != "" else default
    except (TypeError, ValueError):
        return default


# ---------- Customer intelligence ----------


def discover_customer_intel(
    customer_metrics: List[Dict[str, Any]],
    customers: List[Dict[str, Any]],
    transactions: List[Dict[str, Any]],
    top_n_segments: int = 5,
) -> Dict[str, Any]:
    """High-value segments, revenue concentration, retention risk."""
    # Join metrics to customers; aggregate by segment (e.g. Age_Group + Location_Tier)
    metrics_by_customer: Dict[str, Dict[str, Any]] = {}
    for m in customer_metrics:
        cid = m.get("Customer_ID") or m.get("customer_id")
        if cid:
            metrics_by_customer[str(cid)] = m

    customer_ids_with_transactions = set()
    for t in transactions:
        cid = t.get("Customer_ID") or t.get("customer_id")
        if cid:
            customer_ids_with_transactions.add(str(cid))

    # Segment = e.g. Account_Type or Age_Group
    segment_key = "Account_Type" if any(m.get("Account_Type") for m in customer_metrics) else "Age_Group"
    by_segment: Dict[str, Dict[str, Any]] = {}
    for c in customers:
        cid = c.get("Customer_ID") or c.get("customer_id")
        seg = c.get(segment_key) or c.get("Age_Group") or "Unknown"
        if seg not in by_segment:
            by_segment[seg] = {"customers": [], "revenue": 0.0, "clv": 0.0, "retention_risk_sum": 0.0, "count": 0}
        m = metrics_by_customer.get(str(cid), {})
        rev = _float(m.get("Annual_Revenue") or m.get("annual_revenue"))
        clv = _float(m.get("Customer_Lifetime_Value") or m.get("customer_lifetime_value"))
        risk = _float(m.get("Retention_Risk") or m.get("retention_risk"))
        by_segment[seg]["customers"].append(cid)
        by_segment[seg]["revenue"] += rev
        by_segment[seg]["clv"] += clv
        by_segment[seg]["retention_risk_sum"] += risk
        by_segment[seg]["count"] += 1

    total_revenue = sum(s["revenue"] for s in by_segment.values())
    total_clv = sum(s["clv"] for s in by_segment.values())
    high_value_segments = []
    for seg, data in by_segment.items():
        avg_clv = data["clv"] / data["count"] if data["count"] else 0
        rev_pct = (data["revenue"] / total_revenue * 100) if total_revenue else 0
        high_value_segments.append({
            "segment": seg,
            "customer_count": data["count"],
            "total_revenue": round(data["revenue"], 2),
            "revenue_pct": round(rev_pct, 1),
            "avg_clv": round(avg_clv, 2),
            "avg_retention_risk": round(data["retention_risk_sum"] / data["count"], 2) if data["count"] else 0,
        })
    high_value_segments.sort(key=lambda x: x["total_revenue"], reverse=True)
    high_value_segments = high_value_segments[:top_n_segments]

    return {
        "high_value_segments": high_value_segments,
        "total_customers": len(customers),
        "total_revenue": round(total_revenue, 2),
        "segments_with_activity": len(by_segment),
    }


# ---------- Product intelligence ----------


def discover_product_intel(
    product_metrics: List[Dict[str, Any]],
    product_catalog: List[Dict[str, Any]],
    transactions: List[Dict[str, Any]],
    high_margin_min_pct: float = 25.0,
    top_n_products: int = 10,
) -> Dict[str, Any]:
    """High-margin products, adoption counts, product combinations from transactions."""
    product_lookup = {str(p.get("Product_ID") or p.get("product_id")): p for p in product_catalog if p.get("Product_ID") or p.get("product_id")}
    margin_by_id: Dict[str, float] = {}
    for m in product_metrics:
        pid = m.get("Product_ID") or m.get("product_id")
        if pid:
            margin_by_id[str(pid)] = _float(m.get("Profit_Margin") or m.get("profit_margin"), 0.0)

    usage_count: Dict[str, int] = {}
    for t in transactions:
        pid = t.get("Product_ID") or t.get("product_id")
        if pid:
            usage_count[str(pid)] = usage_count.get(str(pid), 0) + 1

    high_margin_products = []
    for pid, margin in margin_by_id.items():
        if margin >= high_margin_min_pct:
            p = product_lookup.get(pid, {})
            high_margin_products.append({
                "product_id": pid,
                "product_type": p.get("Product_Type") or p.get("product_type"),
                "profit_margin_pct": round(margin, 1),
                "usage_count": usage_count.get(pid, 0),
            })
    high_margin_products.sort(key=lambda x: (x["profit_margin_pct"], x["usage_count"]), reverse=True)
    high_margin_products = high_margin_products[:top_n_products]

    return {
        "high_margin_products": high_margin_products,
        "products_with_usage": len(usage_count),
        "total_products": len(product_catalog),
    }


# ---------- Feature intelligence ----------


def discover_feature_intel(
    feature_usage: List[Dict[str, Any]],
    market_signals: List[Dict[str, Any]],
    feature_demand_min_requests: int = 10,
) -> Dict[str, Any]:
    """Feature demand surges (usage + requests), gaps from market signals."""
    feature_usage_count: Dict[str, int] = {}
    for row in feature_usage:
        f = row.get("Feature") or row.get("feature")
        if f:
            feature_usage_count[str(f)] = feature_usage_count.get(str(f), 0) + _int(row.get("Usage_Count") or row.get("usage_count"), 1)

    feature_requests: Dict[str, int] = {}
    for s in market_signals:
        f = s.get("requested_feature")
        if f:
            feature_requests[str(f)] = feature_requests.get(str(f), 0) + 1

    demand_surges = []
    for feat, count in feature_usage_count.items():
        requests = feature_requests.get(feat, 0)
        if requests >= feature_demand_min_requests or count >= feature_demand_min_requests:
            demand_surges.append({"feature": feat, "usage_count": count, "requests": requests})
    demand_surges.sort(key=lambda x: x["usage_count"] + x["requests"], reverse=True)

    gaps = []
    for s in market_signals:
        req = s.get("requested_feature")
        if req and _int(s.get("demand_level") or s.get("demand_level"), 0) >= 1:
            gaps.append({
                "feature": req,
                "demand_level": s.get("demand_level"),
                "competitor_offering": s.get("competitor_offering"),
            })

    return {
        "feature_demand_surges": demand_surges[:15],
        "feature_gaps_from_market": gaps[:15],
    }


# ---------- Trend intelligence ----------


def discover_trend_intel(
    product_adoption_history: List[Dict[str, Any]],
    segment_growth_history: List[Dict[str, Any]],
    feature_demand_history: List[Dict[str, Any]],
    adoption_growth_min_pct: float = 10.0,
    decline_growth_max_pct: float = -5.0,
    segment_growth_min_pct: float = 15.0,
) -> Dict[str, Any]:
    """Emerging/declining products, fast-expanding segments, feature demand acceleration."""
    # Simple approach: group by product/segment/feature, compute min/max/latest; infer trend from growth
    from collections import defaultdict
    prod_by_id: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for row in product_adoption_history:
        pid = row.get("Product_ID") or row.get("product_id")
        if pid:
            prod_by_id[str(pid)].append(row)
    seg_by_name: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for row in segment_growth_history:
        seg = row.get("Segment") or row.get("segment")
        if seg:
            seg_by_name[str(seg)].append(row)
    feat_by_name: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for row in feature_demand_history:
        f = row.get("Feature") or row.get("feature")
        if f:
            feat_by_name[str(f)].append(row)

    def _growth_pct(vals: List[float]) -> Optional[float]:
        if len(vals) < 2:
            return None
        try:
            return (vals[-1] - vals[0]) / vals[0] * 100.0 if vals[0] else None
        except Exception:
            return None

    emerging_products = []
    declining_products = []
    for pid, rows in prod_by_id.items():
        rows_sorted = sorted(rows, key=lambda r: r.get("Date") or r.get("date") or "")
        active = [_int(r.get("Active_Customers") or r.get("active_customers")) for r in rows_sorted]
        g = _growth_pct(active)
        if g is not None:
            if g >= adoption_growth_min_pct:
                emerging_products.append({"product_id": pid, "growth_pct": round(g, 1)})
            elif g <= decline_growth_max_pct:
                declining_products.append({"product_id": pid, "growth_pct": round(g, 1)})
    emerging_products.sort(key=lambda x: x["growth_pct"], reverse=True)
    declining_products.sort(key=lambda x: x["growth_pct"])

    fast_expanding_segments = []
    for seg, rows in seg_by_name.items():
        rows_sorted = sorted(rows, key=lambda r: r.get("Date") or r.get("date") or "")
        cnt = [_int(r.get("Customer_Count") or r.get("customer_count")) for r in rows_sorted]
        g = _growth_pct(cnt)
        if g is not None and g >= segment_growth_min_pct:
            fast_expanding_segments.append({"segment": seg, "growth_pct": round(g, 1)})
    fast_expanding_segments.sort(key=lambda x: x["growth_pct"], reverse=True)

    return {
        "emerging_products": emerging_products[:15],
        "declining_products": declining_products[:15],
        "fast_expanding_segments": fast_expanding_segments[:10],
    }


# ---------- Strategic signals ----------


def discover_strategic_signals(
    bundle_opportunity_signals: List[Dict[str, Any]],
    opportunity_score_min: float = 0.5,
    top_n_bundles: int = 10,
) -> Dict[str, Any]:
    """Cross-sell/upsell and bundle opportunities from precomputed signals."""
    eligible = []
    for row in bundle_opportunity_signals:
        score = _float(row.get("Opportunity_Score") or row.get("opportunity_score"))
        if score >= opportunity_score_min:
            eligible.append({
                "bundle_id": row.get("Bundle_ID") or row.get("bundle_id"),
                "product_a": row.get("Product_A") or row.get("product_a"),
                "product_b": row.get("Product_B") or row.get("product_b"),
                "product_c": row.get("Product_C") or row.get("product_c"),
                "segment": row.get("Segment") or row.get("segment"),
                "observed_customers": _int(row.get("Observed_Customers") or row.get("observed_customers")),
                "avg_revenue_per_customer": _float(row.get("Avg_Revenue_Per_Customer") or row.get("avg_revenue_per_customer")),
                "opportunity_score": round(score, 2),
            })
    eligible.sort(key=lambda x: (x["opportunity_score"], x["observed_customers"]), reverse=True)
    top_bundles = eligible[:top_n_bundles]

    return {
        "bundle_opportunities": top_bundles,
        "total_signals_above_threshold": len(eligible),
    }


# ---------- Run all discovery ----------


def run_all_discovery(
    data: Dict[str, Any],
    adoption_growth_min_pct: float,
    decline_growth_max_pct: float,
    high_margin_min_pct: float,
    opportunity_score_min: float,
    segment_growth_min_pct: float,
    feature_demand_min_requests: int,
    top_n_bundles: int,
    top_n_segments: int,
    top_n_products: int,
) -> Dict[str, Dict[str, Any]]:
    """Run all discovery layers and return customer_intel, product_intel, feature_intel, trend_intel, strategic_signals."""
    customer_intel = discover_customer_intel(
        data.get("customer_metrics", []),
        data.get("customers", []),
        data.get("transactions", []),
        top_n_segments=top_n_segments,
    )
    product_intel = discover_product_intel(
        data.get("product_metrics", []),
        data.get("product_catalog", []),
        data.get("transactions", []),
        high_margin_min_pct=high_margin_min_pct,
        top_n_products=top_n_products,
    )
    feature_intel = discover_feature_intel(
        data.get("feature_usage", []),
        data.get("market_signals", []),
        feature_demand_min_requests=feature_demand_min_requests,
    )
    trend_intel = discover_trend_intel(
        data.get("product_adoption_history", []),
        data.get("segment_growth_history", []),
        data.get("feature_demand_history", []),
        adoption_growth_min_pct=adoption_growth_min_pct,
        decline_growth_max_pct=decline_growth_max_pct,
        segment_growth_min_pct=segment_growth_min_pct,
    )
    strategic_signals = discover_strategic_signals(
        data.get("bundle_opportunity_signals", []),
        opportunity_score_min=opportunity_score_min,
        top_n_bundles=top_n_bundles,
    )
    return {
        "customer_intel": customer_intel,
        "product_intel": product_intel,
        "feature_intel": feature_intel,
        "trend_intel": trend_intel,
        "strategic_signals": strategic_signals,
    }
